# Data preprocessing
Reads the supplied source workbooks without changing them. Their column labels are text-corrupted, so field names are assigned by verified column position. Duplicate drug names are removed and missing numeric drug fields are median-imputed. Derived tables are saved separately from existing outputs.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
ROOT=Path("..").resolve()
RAW=ROOT/"data/raw"; PROC=ROOT/"data/processed/final"
FIG=ROOT/"results/final/figures"; TAB=ROOT/"results/final/tables"
for p in (PROC,FIG,TAB): p.mkdir(parents=True,exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["savefig.dpi"]=300

In [2]:
drug=pd.read_excel(RAW/"drug_data.xlsx"); mn=pd.read_excel(RAW/"mn_data.xlsx")
drug_cols=["Drug","Category","MW_g_mol","LogP","Solubility_mg_mL","pKa","Kp_cm_h","Flux_mg_cm2_h","Class"]
mn_cols=["Design_ID","Length_um","Base_size_um","Tip_diameter_um","Inner_diameter_um","Cone_radius_um","Density_cm2","Insertion_speed_mm_s","Concentration_pct","Penetration_efficiency","Quality"]
if len(drug.columns)!=len(drug_cols) or len(mn.columns)!=len(mn_cols): raise ValueError("Workbook schema differs from inspected source.")
drug.columns=drug_cols; mn.columns=mn_cols
drug=drug.drop_duplicates("Drug",keep="first").copy()
for c in ["MW_g_mol","LogP","Solubility_mg_mL","pKa","Kp_cm_h","Flux_mg_cm2_h"]:
    drug[c]=pd.to_numeric(drug[c],errors="coerce")
    if drug[c].notna().any(): drug[c]=drug[c].fillna(drug[c].median())
for c in ["Length_um","Base_size_um","Tip_diameter_um","Inner_diameter_um","Cone_radius_um","Density_cm2","Insertion_speed_mm_s","Concentration_pct"]:
    mn[c]=pd.to_numeric(mn[c],errors="coerce")
if mn[["Design_ID","Length_um","Base_size_um","Tip_diameter_um","Inner_diameter_um","Cone_radius_um","Density_cm2","Insertion_speed_mm_s","Concentration_pct"]].isna().any().any(): raise ValueError("Missing required design data.")
drug.to_csv(PROC/"clean_drug_data.csv",index=False,encoding="utf-8-sig")
mn.to_csv(PROC/"clean_mn_data.csv",index=False,encoding="utf-8-sig")
print("Unique drugs:",len(drug),"design rows:",len(mn),"unique designs:",mn[["Length_um","Base_size_um","Tip_diameter_um","Inner_diameter_um","Cone_radius_um","Density_cm2","Insertion_speed_mm_s","Concentration_pct"]].drop_duplicates().shape[0])
for c in ["Length_um","Base_size_um","Tip_diameter_um","Inner_diameter_um","Cone_radius_um","Density_cm2","Insertion_speed_mm_s","Concentration_pct"]: print(c,sorted(mn[c].unique()))
print("MW range:",drug.MW_g_mol.min(),drug.MW_g_mol.max())

Unique drugs: 26 design rows: 144 unique designs: 144
Length_um [np.int64(350), np.int64(450), np.int64(550), np.int64(650)]
Base_size_um [np.int64(100), np.int64(300)]
Tip_diameter_um [np.float64(1.44), np.float64(1.93), np.float64(2.42)]
Inner_diameter_um [np.int64(50), np.int64(150)]
Cone_radius_um [np.float64(2.88), np.float64(3.86), np.float64(4.84)]
Density_cm2 [np.int64(150), np.int64(500), np.int64(900)]
Insertion_speed_mm_s [np.int64(5)]
Concentration_pct [np.float64(0.5)]
MW range: 131.13 6046.0
